## Principios básicos del prompting

1. Los prompts deben ser sencillos y claros. No es necesario escribir un párrafo entero para obtener una respuesta.
2. Dale al modelo tiempo para "pensar" en la respuesta. No le des un prompt y esperes una respuesta inmediata.

In [1]:
import openai
import os
from IPython.display import Markdown
import requests
import json

from dotenv import load_dotenv
load_dotenv()

openai.api_key = os.getenv("OPENAI_API_KEY")
auth_token = os.getenv("AUTH_TOKEN")


headers = {
  'Content-Type': 'application/json',
  'Authorization': f"Bearer {auth_token}"
}
url = 'https://api.awanllm.com/v1/chat/completions' # URL for the API

In [2]:
# Función para llamar a la API de LLMCloud para preguntarle a LLAMA
def preguntar_llama(sys_prompt, prompt):
    payload = json.dumps({
        "model": "Meta-Llama-3-8B-Instruct",
        "messages": [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt}
        ],
      })
    return requests.request("POST", url, headers=headers, data=payload).json()['choices'][0]['message']['content']


In [3]:
client = openai.OpenAI()

def get_completion(prompt, model="gpt-3.5-turbo-0125"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.1,
        max_tokens=256,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )
    return response.choices[0].message.content

### Tácticas

1. Utiliza delimitadores para indicar cómo distinguir partes del input.
    Delimitadores pueden ser ```, """, < >, `<tag> </tag>`, etc.

In [4]:
# Definimos el texto
text = """En esencia, a menor temperatura en un Modelo de Lenguaje de Gran Escala (LLM), los resultados tienden a ser más deterministas. Esto implica que el modelo optará con más frecuencia por el siguiente token de mayor probabilidad. Incrementar la temperatura introduce mayor variabilidad en las respuestas, lo que promueve resultados más variados o creativos. Básicamente, al elevar la temperatura, se incrementa la posibilidad de elegir diferentes tokens como opciones. Desde un punto de vista práctico, en tareas como preguntas y respuestas basadas en información objetiva, se sugiere utilizar un valor de temperatura reducido para lograr respuestas más certeras y breves. Sin embargo, en la creación de poesía o en otras actividades creativas, puede resultar ventajoso aumentar el valor de la temperatura. El aumento en la temperatura puede generar alucinaciones."""

# Definimos el prompt
prompt = f"Resume el texto delimitado por las triples comillas en una sola oración: {text}"

# Obtenemos la respuesta
response = get_completion(prompt)
display(Markdown(f"**Respuesta GPT:** {response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}"))

**Respuesta GPT:** A menor temperatura en un Modelo de Lenguaje de Gran Escala (LLM) los resultados tienden a ser más deterministas, optando por el siguiente token de mayor probabilidad, mientras que incrementar la temperatura introduce mayor variabilidad en las respuestas, promoviendo resultados más variados o creativos al incrementar la posibilidad de elegir diferentes tokens como opciones, siendo recomendado un valor de temperatura reducido para respuestas certeras y breves en tareas como preguntas y respuestas basadas en información objetiva, pero ventajoso aumentar el valor de la temperatura en la creación de poesía u otras actividades creativas, aunque el aumento en la temperatura puede generar alucinaciones.


---------------------------------------------



**Respuesta de LLAMA:** En un Modelo de Lenguaje de Gran Escala, una temperatura baja produce resultados más deterministas y predeterminados, mientras que una temperatura alta introduce variabilidad y creatividad en las respuestas, siendo recomendable utilizar temperaturas bajas para preguntas y respuestas objetivas y altas para creación literaria.

2. JSON, HTML, XML, YAML, etc. son formatos que el modelo puede entender y que pueden ayudar a estructurar el input.

In [5]:
prompt = """Genera una lista de tres títulos de libros inventados junto con sus autores y géneros. Proporciónelos en formato JSON con las siguientes claves: book_id, título, autor, género"""
response = get_completion(prompt)

print(f"**Respuesta GPT:** {response}")
print("\n---------------------------------------------\n")
print(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}")

**Respuesta GPT:** {
    "books": [
        {
            "book_id": 1,
            "título": "El misterio de la isla perdida",
            "autor": "María García",
            "género": "Misterio"
        },
        {
            "book_id": 2,
            "título": "El secreto de la montaña encantada",
            "autor": "Juan Pérez",
            "género": "Aventura"
        },
        {
            "book_id": 3,
            "título": "La magia de la noche estrellada",
            "autor": "Ana López",
            "género": "Fantasía"
        }
    ]
}

---------------------------------------------

**Respuesta de LLAMA:** Aquí tienes la lista de títulos de libros inventados:

```
[
  {
    "book_id": "1",
    "título": "El misterio del jardín de cristal",
    "autor": "Ana María García",
    "género": "Misterio"
  },
  {
    "book_id": "2",
    "título": "La ciudad de los sueños perdidos",
    "autor": "Juan Carlos Rodríguez",
    "género": "Ciencia Ficción"
  },
  {
    "book_id":

3. Pídele al modelo que verifique si se están cumpliendo ciertas condiciones.

In [6]:
text_1 = """Preparar una exquisita taza de café en una máquina espresso es un proceso sencillo. En primer lugar, asegúrate de que la máquina esté encendida y lista para usar. Luego, muele los granos de café según el grosor deseado y colócalos en el portafiltro. Compacta el café con el tampador para garantizar una extracción uniforme. Ajusta la cantidad de café según tu preferencia y coloca el portafiltro en la máquina. Activa la máquina y espera a que el café se extraiga lentamente, disfrutando del aroma que llena el aire. ¡Y voilà! Disfruta de tu aromático y delicioso café espresso."""

prompt = f"""
Vas a recibir un texto delimitado por triples comillas.
Si contiene una secuencia de instrucciones, reescribe esas instrucciones en el siguiente formato:

Paso 1 - ...\n
Paso 2 - ...\n
…
Paso N - ... \n

Si el texto no contiene una secuencia de instrucciones, entonces simplemente escribe "No se proporcionaron pasos."

Texto:
```{text_1}```
"""

response = get_completion(prompt)
display(Markdown(f"**Respuesta GPT:** {response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}"))

**Respuesta GPT:** Paso 1 - Asegurarse de que la máquina esté encendida y lista para usar.

Paso 2 - Moler los granos de café según el grosor deseado y colocarlos en el portafiltro.

Paso 3 - Compactar el café con el tampador para garantizar una extracción uniforme.

Paso 4 - Ajustar la cantidad de café según la preferencia y colocar el portafiltro en la máquina.

Paso 5 - Activar la máquina y esperar a que el café se extraiga lentamente, disfrutando del aroma que llena el aire.

Paso 6 - ¡Y voilà! Disfrutar de tu aromático y delicioso café espresso.


---------------------------------------------



**Respuesta de LLAMA:** Paso 1 - Asegúrate de que la máquina esté encendida y lista para usar.

Paso 2 - Muele los granos de café según el grosor deseado y colócalos en el portafiltro.

Paso 3 - Compacta el café con el tampador para garantizar una extracción uniforme.

Paso 4 - Ajusta la cantidad de café según tu preferencia y coloca el portafiltro en la máquina.

Paso 5 - Activa la máquina y espera a que el café se extraiga lentamente, disfrutando del aroma que llena el aire.

(Note: No hay más pasos después de este)

4. Few-shot prompting: Utiliza ejemplos para que el modelo entienda mejor lo que quieres.

In [11]:
text_1 = """Los humanos son seres sociales por naturaleza y, por lo tanto, la comunicación es una parte esencial de la vida cotidiana. La comunicación efectiva puede ser verbal o no verbal, y ambas formas son igualmente importantes. Algunos estudios sugieren que el 93% de la comunicación es no verbal, lo que significa que el lenguaje corporal, las expresiones faciales y otros gestos juegan un papel crucial en la forma en que nos comunicamos. Por otro lado, la comunicación verbal también es fundamental, ya que nos permite expresar nuestros pensamientos, sentimientos y emociones de manera clara y concisa."""

prompt = f"""
Vas a recibir un texto delimitado por triples comillas.
Si contiene una secuencia de instrucciones, reescribe esas instrucciones en el siguiente formato:

Paso 1 - ...\n
Paso 2 - ...\n
…
Paso N - ... \n

Si el texto no contiene una secuencia de instrucciones, entonces simplemente escribe "No se proporcionaron pasos."

Texto:
```{text_1}```
"""

response = get_completion(prompt)
display(Markdown(f"**Respuesta GPT:** {response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}"))

**Respuesta GPT:** No se proporcionaron pasos.


---------------------------------------------



**Respuesta de LLAMA:** No se proporcionan pasos.

In [7]:
prompt = """
Tu tarea es responder de manera consistente.

<hijo>: Enséñame sobre la paciencia.

<abuelo>: El río que talla el valle más profundo fluye desde una modesta fuente; la sinfonía más grandiosa se origina a partir de una sola nota; el tapiz más intrincado comienza con un hilo solitario.

<hijo>: Enséñame sobre la resiliencia.
"""
response = get_completion(prompt)
display(Markdown(f"**Respuesta GPT:** {response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}"))

**Respuesta GPT:** <abuelo>: La resiliencia es como el bambú, que se dobla con el viento pero nunca se quiebra. Es la capacidad de enfrentar los desafíos con valentía y perseverancia, de levantarse una y otra vez después de cada caída. Es la fuerza interior que nos permite seguir adelante, incluso en los momentos más difíciles. Recuerda siempre que, al igual que el bambú, tú también puedes doblarte pero nunca romperte.


---------------------------------------------



**Respuesta de LLAMA:** Abuelo, veo que mi hijo te ha pedido algo nuevo y emocionante. ¡Vamos allá!

<abuelo>: La resiliencia es como el tronco de un árbol viejo, erosionado por las tormentas del tiempo. A medida que las tempestades golpean y golpean, el tronco se vuelve más ancho y más resistente, porque cada vez que cae una hoja o se desprenden ramas, el árbol reacciona creando nuevas defensas para protegerse. De igual manera, cuando la adversidad nos golpea, nuestra resiliencia se fortalece, permitiendo que nos adaptemos y crezcamos en la adversidad. Así que no temas, hijo mío, que la resiliencia no es la ausencia de dificultades, sino la capacidad de superarlas y seguir adelante, más fuerte y sabio.

Principio 2: Darle tiempo al modelo para "pensar" en la respuesta.

Táctica: Especifica los pasos requeridos para completar la tarea.

In [8]:
text = """
En un tranquilo bosque, los hermanos Mario y Sara se aventuraron en busca de un antiguo tesoro perdido. Mientras caminaban entre los árboles, riendo y compartiendo historias, un ruido extraño los detuvo de repente. MArio, intrigado, se acercó para investigar y accidentalmente activó una trampa oculta. Una red se desplegó atrapándolo, y Sara, al intentar liberarlo, cayó en un foso cercano.

Afortunadamente, ninguno de los dos resultó gravemente herido. Juntos, encontraron la manera de salir de sus respectivas trampas y regresaron a casa con algunas raspaduras y moretones. Sin embargo, en lugar de desanimarse, se abrazaron con alivio y se rieron de su mala suerte.

Decidieron que la próxima vez serían más cuidadosos, pero su determinación por descubrir el tesoro perdido no disminuyó. Con una nueva perspectiva sobre la aventura, continuaron explorando el bosque con entusiasmo renovado, emocionados por lo que el destino les depararía.
"""

prompt_1 = f"""
Realiza las siguientes acciones: 
1 - Resumir el siguiente texto delimitado por triple comillas con 1 frase.
2 - Traduzca el resumen al francés.
3 - Listar cada nombre en el resumen en francés.
4 - Generar un objeto json que contenga lo siguiente
claves: french_summary, num_names.

Separa tus respuestas con saltos de línea.

Texto:
```{text}```
"""
response = get_completion(prompt_1)
print(f"**Respuesta GPT para el prompt 1:** {response}")
print("\n---------------------------------------------\n")
print(f"**Respuesta de LLAMA para el prompt 1:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt_1)}")

**Respuesta GPT para el prompt 1:** 1 - Mario y Sara se aventuran en busca de un tesoro perdido en un bosque tranquilo, pero terminan atrapados en trampas.

2 - Mario et Sara se lancent à la recherche d'un trésor perdu dans une forêt tranquille, mais se retrouvent piégés.

3 - Mario, Sara

4 - {
    "french_summary": "Mario et Sara se lancent à la recherche d'un trésor perdu dans une forêt tranquille, mais se retrouvent piégés.",
    "num_names": 2
}

---------------------------------------------

**Respuesta de LLAMA para el prompt 1:** **Resumen del texto**

Los hermanos Mario y Sara salen a buscar un tesoro en un bosque y accidentalmente activan una trampa, pero afortunadamente logran escapar sin graves daños y deciden seguir adelante con su búsqueda.

**Traducción al francés**

Les frères Mario et Sara partent à la recherche d'un trésor dans une forêt et accidentellement déclenchent une piège, mais heureusement ils parviennent à s'en échapper sans grands dommages et décident de con

Pide el resultado en un formato específico.

In [9]:
prompt_2 = f"""
tu tarea consiste en realizar las siguientes acciones: 
1 - Resumir el siguiente texto delimitado por <> con 1 frase.
2 - Traduzcir el resumen al francés.
3 - Listar cada nombre en el resumen en francés.
4 - Dar salida a un objeto json que contenga las siguientes claves: french_summary, num_names.

Utilice el siguiente formato:
Texto: <texto a resumir>
Resumen: <resumen>
Traducción: <traducción del resumen>
Nombres: <lista de nombres en resumen>
JSON de salida: <json con resumen y número_nombres>

Texto: <{text}>
"""
response = get_completion(prompt_2)
print(f"**Respuesta GPT para el prompt 2:** {response}")
print("\n---------------------------------------------\n")
print(f"**Respuesta de LLAMA para el prompt 2:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt_2)}")

**Respuesta GPT para el prompt 2:** Resumen: En un bosque tranquilo, los hermanos Mario y Sara se aventuran en busca de un tesoro perdido, pero terminan atrapados en trampas antes de regresar a casa con raspaduras y moretones, decididos a seguir con su búsqueda.

Traducción: Dans une forêt tranquille, les frères Mario et Sara s'aventurent à la recherche d'un trésor perdu, mais se retrouvent piégés avant de rentrer chez eux avec des égratignures et des bleus, déterminés à poursuivre leur quête.

Nombres: Mario, Sara

JSON de salida: {"french_summary": "Dans une forêt tranquille, les frères Mario et Sara s'aventurent à la recherche d'un trésor perdu, mais se retrouvent piégés avant de rentrer chez eux avec des égratignures et des bleus, déterminés à poursuivre leur quête.", "num_names": 2}

---------------------------------------------

**Respuesta de LLAMA para el prompt 2:** Texto: <En un tranquilo bosque, los hermanos Mario y Sara se aventuraron en busca de un antiguo tesoro perdido.>

Táctica 2: Pedir al modelo que elabore su propia solución antes de llegar a una conclusión precipitada.

## Limitaciones del modelo: Alucinaciones

Apple es una compañía de tecnología real pero iCan no es un producto real de Apple. Sin embargo, el modelo puede alucinar que iCan es un producto real de Apple y devolvernos una respuesta que no es cierta.

In [10]:
prompt = """
Háblame sobre el iCan, la última innovación en tecnología de dispositivos portátiles de Apple.
"""
response = get_completion(prompt)
display(Markdown(f"**Respuesta GPT:** {response}"))
print("\n---------------------------------------------\n")
display(Markdown(f"**Respuesta de LLAMA:** {preguntar_llama('Eres un asistente de IA cuyo trabajo es responder a lo que te está pidiendo el usuario de la forma más fiel al prompt', prompt=prompt)}"))

**Respuesta GPT:** El iCan es el último dispositivo portátil de Apple que combina las funciones de un reloj inteligente, un monitor de actividad física y un asistente personal en un solo dispositivo. Con un diseño elegante y moderno, el iCan cuenta con una pantalla táctil de alta resolución que muestra notificaciones, mensajes, llamadas y otras funciones de forma clara y fácil de usar.

Además, el iCan incluye sensores avanzados que monitorean la actividad física del usuario, como el ritmo cardíaco, la calidad del sueño, los pasos dados y las calorías quemadas. También cuenta con funciones de asistente personal que permiten al usuario realizar tareas como enviar mensajes, hacer llamadas, programar recordatorios y controlar dispositivos inteligentes en el hogar.

El iCan se sincroniza con la aplicación Health de Apple, que permite al usuario llevar un registro de su actividad física, establecer metas de salud y recibir recomendaciones personalizadas para mejorar su bienestar. En resumen, el iCan es una innovación revolucionaria en tecnología de dispositivos portátiles que combina estilo, funcionalidad y conven


---------------------------------------------



**Respuesta de LLAMA:** Disculpa, pero no hay un dispositivo llamado "iCan" lanzado por Apple. Es posible que estés pensando en otro producto o servicio de Apple o confundiendo información.

Sin embargo, Apple ha lanzado varios productos y servicios recientemente, como el iPhone 13, el iPad Air (2022), el MacBook Air M1, el Apple Watch Series 7, entre otros. También ha mejorado y actualizado sus servicios como iCloud, Apple Music, Apple TV+, etc.

Si necesitas información específica sobre algún producto o servicio de Apple, estaré feliz de ayudarte. ¡No dudes en preguntar!

## Tu turno de experimentar!
